In [0]:
%pip install confluent-kafka  # required by job cluster until we deploy via DABs

In [0]:
import json
import logging
import uuid
from datetime import datetime, timezone
from confluent_kafka import Producer, KafkaException

In [0]:
logger = logging.getLogger("DatabricksWorkflow")
logger.setLevel(logging.INFO)
handler = logging.StreamHandler()
formatter = logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s')
handler.setFormatter(formatter)
if not logger.hasHandlers():
    logger.addHandler(handler)

In [0]:
config_path = "dbfs:/configs/config.json"
try:
    config = spark.read.option("multiline", "true").json(config_path)
    logger.info(f"Successfully read config file from {config_path}")
except Exception as e:
    logger.error(f"Could not read config file at {config_path}: {e}", exc_info=True)
    raise FileNotFoundError(f"Could not read config file at {config_path}: {e}")

try:
    first_row = config.first()
    env = first_row["env"].strip().lower()
    lz_key = first_row["lz_key"].strip().lower()
    logger.info(f"Extracted configs: env={env}, lz_key={lz_key}")
except Exception as e:
    logger.error(f"Missing expected keys 'env' or 'lz_key' in config file: {e}", exc_info=True)
    raise KeyError(f"Missing expected keys 'env' or 'lz_key' in config file: {e}")

try:
    keyvault_name = f"ingest{lz_key}-meta002-{env}"
    logger.info(f"Constructed keyvault name: {keyvault_name}")
except Exception as e:
    logger.error(f"Error constructing keyvault name: {e}", exc_info=True)
    raise ValueError(f"Error constructing keyvault name: {e}")

In [0]:
try:
    client_secret = dbutils.secrets.get(scope=keyvault_name, key='SERVICE-PRINCIPLE-CLIENT-SECRET')
    logger.info("Successfully retrieved SERVICE-PRINCIPLE-CLIENT-SECRET from Key Vault")
except Exception as e:
    logger.error(f"Could not retrieve 'SERVICE-PRINCIPLE-CLIENT-SECRET' from Key Vault '{keyvault_name}': {e}", exc_info=True)
    raise KeyError(f"Could not retrieve 'SERVICE-PRINCIPLE-CLIENT-SECRET' from Key Vault '{keyvault_name}': {e}")

try:
    tenant_id = dbutils.secrets.get(scope=keyvault_name, key='SERVICE-PRINCIPLE-TENANT-ID')
    logger.info("Successfully retrieved SERVICE-PRINCIPLE-TENANT-ID from Key Vault")
except Exception as e:
    logger.error(f"Could not retrieve 'SERVICE-PRINCIPLE-TENANT-ID' from Key Vault '{keyvault_name}': {e}", exc_info=True)
    raise KeyError(f"Could not retrieve 'SERVICE-PRINCIPLE-TENANT-ID' from Key Vault '{keyvault_name}': {e}")

try:
    client_id = dbutils.secrets.get(scope=keyvault_name, key='SERVICE-PRINCIPLE-CLIENT-ID')
    logger.info("Successfully retrieved SERVICE-PRINCIPLE-CLIENT-ID from Key Vault")
except Exception as e:
    logger.error(f"Could not retrieve 'SERVICE-PRINCIPLE-CLIENT-ID' from Key Vault '{keyvault_name}': {e}", exc_info=True)
    raise KeyError(f"Could not retrieve 'SERVICE-PRINCIPLE-CLIENT-ID' from Key Vault '{keyvault_name}': {e}")

logger.info("\u2705 Successfully retrieved all Service Principal secrets from Key Vault")

In [0]:
curated_storage_account = f"ingest{lz_key}curated{env}"

for storage_account in [curated_storage_account]:
    try:
        configs = {
            f"fs.azure.account.auth.type.{storage_account}.dfs.core.windows.net": "OAuth",
            f"fs.azure.account.oauth.provider.type.{storage_account}.dfs.core.windows.net":
                "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
            f"fs.azure.account.oauth2.client.id.{storage_account}.dfs.core.windows.net": client_id,
            f"fs.azure.account.oauth2.client.secret.{storage_account}.dfs.core.windows.net": client_secret,
            f"fs.azure.account.oauth2.client.endpoint.{storage_account}.dfs.core.windows.net":
                f"https://login.microsoftonline.com/{tenant_id}/oauth2/token"
        }
        for key, val in configs.items():
            spark.conf.set(key, val)
        logger.info(f"\u2705 Successfully configured OAuth for storage account: {storage_account}")
    except Exception as e:
        logger.error(f"Error configuring OAuth for storage account '{storage_account}': {e}", exc_info=True)
        raise RuntimeError(f"Error configuring OAuth for storage account '{storage_account}': {e}")

In [0]:
dbutils.widgets.dropdown(
    name="stage",
    defaultValue="CCD",
    choices=["CCD", "CDAM", "CASE_LINKING"],
    label="Stage"
)
stage = dbutils.widgets.get("mvp")

In [0]:
eh_kv_secret = dbutils.secrets.get(scope=keyvault_name, key="RootManageSharedAccessKey")

eventhubs_hostname = f"ingest{lz_key}-integration-eventHubNamespace001-{env}.servicebus.windows.net:9093"

RESULTS_EH_MAP = {
    "CCD":          f"evh-active-res-{env}-{lz_key}-uks-dlrm-01",
    "CDAM":         f"evh-active-cdam-res-{env}-{lz_key}-uks-dlrm-01",
    "CASE_LINKING": f"evh-active-caselink-res-{env}-{lz_key}-uks-dlrm-01",
}

results_eh_topic = RESULTS_EH_MAP[stage]
logger.info(f"Results Event Hub topic: {results_eh_topic}")

kafka_conf = {
    'bootstrap.servers': eventhubs_hostname,
    'security.protocol': 'SASL_SSL',
    'sasl.mechanism': 'PLAIN',
    'sasl.username': '$ConnectionString',
    'sasl.password': eh_kv_secret,
    'retries': 5,
}

In [0]:
current_date_time = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S.%f")

# Uncomment payload to use

# CCD results audit schema:
payload = {
    "RunID":           "PLACEHOLDER_RUN_ID",
    "AriaCaseNo":      "AA/12345/2024",
    "CaseNo":          "APPEALS_AA_12345_2024.json",
    "State":           "listing",
    "StartDateTime":   current_date_time,
    "EndDateTime":     current_date_time,
    "CCDCaseID":       "PLACEHOLDER_CCD_CASE_ID",
    "Status":          "SUCCESS",
    "SuccessResponse": "",
    "Error":           "",
    "StartResponse":   ""
}

# CDAM results audit schema:
# payload = {
#     "RunID":         "PLACEHOLDER_RUN_ID",
#     "CaseNo":        "AA/12345/2024",
#     "StartDateTime": current_date_time,
#     "EndDateTime":   current_date_time,
#     "Status":        "SUCCESS",
#     "Error":         "",
#     "CDAMResponse":  ""
# }

# CASE_LINKING results audit schema:
# payload = {
#     "RunID":                  "PLACEHOLDER_RUN_ID",
#     "CCDCaseReferenceNumber": "PLACEHOLDER_CCD_CASE_REF",
#     "CaseLinkCount":          "1",
#     "StartDateTime":          current_date_time,
#     "EndDateTime":            current_date_time,
#     "Status":                 "SUCCESS",
#     "Error":                  ""
# }


print(json.dumps(payload, indent=2))

In [0]:
MESSAGE_KEY_MAP = {
    "CCD":          payload.get("CaseNo"),
    "CDAM":         payload.get("CaseNo"),
    "CASE_LINKING": payload.get("CCDCaseReferenceNumber"),
}

message_key = MESSAGE_KEY_MAP[stage]

delivery_results = []

def delivery_report(err, msg):
    if err:
        delivery_results.append(("ERROR", str(err)))
        logger.error(f"Message delivery failed: {err}")
    else:
        delivery_results.append(("SUCCESS", ""))
        logger.info(f"Message delivered to {msg.topic()} [{msg.partition()}]")

producer = Producer(kafka_conf)
try:
    producer.produce(
        topic=results_eh_topic,
        key=message_key.encode("utf-8") if message_key else None,
        value=json.dumps(payload).encode("utf-8"),
        callback=delivery_report
    )
    producer.flush()
    logger.info(f"Published result to {results_eh_topic}")
except KafkaException as e:
    logger.error(f"Kafka produce failed: {e}")
    raise

for status, error in delivery_results:
    if status == "ERROR":
        print(f"Delivery failed: {error}")
    else:
        print(f"Successfully delivered to {results_eh_topic}")

In [0]:
dbutils.notebook.exit(f"{stage} result republished successfully")
logger.info(f"Completed republish for MVP: {stage}")